In [11]:
import math
from datetime import datetime, timedelta
from typing import Optional, Union, List, Dict
import xml.etree.ElementTree as ET


def meters_per_deg_lat(lat_deg: float) -> float:
    """Meters per degree of latitude (WGS84 approximation)."""
    phi = math.radians(lat_deg)
    return (
        111132.92
        - 559.82 * math.cos(2 * phi)
        + 1.175 * math.cos(4 * phi)
        - 0.0023 * math.cos(6 * phi)
    )


def meters_per_deg_lon(lat_deg: float) -> float:
    """Meters per degree of longitude (WGS84 approximation)."""
    phi = math.radians(lat_deg)
    return (
        111412.84 * math.cos(phi)
        - 93.5 * math.cos(3 * phi)
        + 0.118 * math.cos(5 * phi)
    )


def _numbered_filename(base: str, idx: int) -> str:
    """Insert .{idx} before the extension; if no extension, append .{idx}.gpx"""
    if "." in base:
        stem, ext = base.rsplit(".", 1)
        return f"{stem}.{idx}.{ext}"
    else:
        return f"{base}.{idx}.gpx"


def generate_square_gpx_fast_vertical(
    start_lat: float = 22.198745,   # Default: Macau approx center
    start_lon: float = 113.543873,
    west_m: float = 0.0,
    north_m: float = 0.0,
    east_m: float = 100.0,
    south_m: float = 100.0,
    horizontal_step_m: float = 1.0,   # eastward step between columns
    secs_per_meter: float = 1.0,      # time pacing (s per meter)
    filename: str = "square_fast_vertical.gpx",
    start_time_utc: Optional[datetime] = None,  # <-- Python 3.8 compatible
    max_columns: int = 100_000,       # safety cap on number of columns
    allow_exceed_east: bool = True,   # last east step can slightly exceed east boundary
    # --- NEW: optional tiling limits (meters) ---
    max_tile_width_m: Optional[float] = None,
    max_tile_height_m: Optional[float] = None,
) -> Union[Dict, List[Dict]]:
    """
    Generate a GPX track that sweeps a rectangle:
      - Corner UL = start shifted west and north by (west_m, north_m)
      - Corner LR = start shifted east and south by (east_m, south_m)

    Pattern:
      1) Start at UL
      2) One big vertical move to the opposite north/south boundary
      3) Step east by ~horizontal_step_m
      4) One big vertical move back
      5) Repeat until east boundary is reached (or slightly exceeded if allowed)

    NEW (optional):
      If max_tile_width_m and/or max_tile_height_m are set and the requested
      rectangle exceeds the limit(s), it is automatically split into tiles
      (from top-left to bottom-right), and multiple files are written:
        base.1.gpx, base.2.gpx, ...

      Each tile uses the same sweeping pattern and is generated by a recursive
      call with the tile's center as the local start position (to keep all
      offsets non-negative).
    """
    # Validate inputs
    for name, v in {
        "west_m": west_m, "north_m": north_m, "east_m": east_m, "south_m": south_m,
        "horizontal_step_m": horizontal_step_m, "secs_per_meter": secs_per_meter
    }.items():
        if v < 0:
            raise ValueError(f"{name} must be non-negative.")

    # Prepare time
    if start_time_utc is None:
        start_time_utc = datetime.utcnow()

    # Compute UL and LR corners (use local scale at start latitude; accurate for local rectangles)
    m_per_deg_lat0 = meters_per_deg_lat(start_lat)
    m_per_deg_lon0 = meters_per_deg_lon(start_lat)

    lat_ul = start_lat + (north_m / m_per_deg_lat0)
    lon_ul = start_lon - (west_m / m_per_deg_lon0)
    lat_lr = start_lat - (south_m / m_per_deg_lat0)
    lon_lr = start_lon + (east_m / m_per_deg_lon0)

    if not (lat_ul > lat_lr):
        raise ValueError("Computed UL must be north of LR. Check north_m/south_m.")
    if not (lon_ul < lon_lr):
        raise ValueError("Computed UL must be west of LR. Check west_m/east_m.")

    # Geometry estimates (use mid-lat for slightly better scale)
    mid_lat = (lat_ul + lat_lr) / 2.0
    width_m_est = (lon_lr - lon_ul) * meters_per_deg_lon(mid_lat)
    height_m_est = (lat_ul - lat_lr) * meters_per_deg_lat(mid_lat)

    # Also compute width/height in meters under the same local scale used
    # for converting offsets originally (start_lat), to drive tiling boundaries consistently.
    width_m_from_start = (lon_lr - lon_ul) * m_per_deg_lon0
    height_m_from_start = (lat_ul - lat_lr) * m_per_deg_lat0

    # If tiling is required, split and recurse per tile
    needs_tiling = (
        (max_tile_width_m is not None and width_m_from_start > max_tile_width_m) or
        (max_tile_height_m is not None and height_m_from_start > max_tile_height_m)
    )

    if needs_tiling:
        # Default infinite size if one dimension limit is None
        tile_w = max_tile_width_m if max_tile_width_m is not None else width_m_from_start
        tile_h = max_tile_height_m if max_tile_height_m is not None else height_m_from_start

        # Compute number of tiles horizontally and vertically
        n_cols = int(math.ceil(width_m_from_start / max(1e-9, tile_w)))
        n_rows = int(math.ceil(height_m_from_start / max(1e-9, tile_h)))

        pieces: List[Dict] = []
        piece_idx = 1

        for r in range(n_rows):
            # Row vertical boundaries in degrees (north-to-south)
            top_offset_m = r * tile_h
            # For the last row, clamp to rectangle bottom
            this_row_h_m = min(tile_h, height_m_from_start - top_offset_m)

            lat_top = lat_ul - (top_offset_m / m_per_deg_lat0)
            lat_bottom = lat_top - (this_row_h_m / m_per_deg_lat0)

            for c in range(n_cols):
                # Column horizontal boundaries in degrees (west-to-east)
                left_offset_m = c * tile_w
                this_col_w_m = min(tile_w, width_m_from_start - left_offset_m)

                lon_left = lon_ul + (left_offset_m / m_per_deg_lon0)
                lon_right = lon_left + (this_col_w_m / m_per_deg_lon0)

                # Center of tile
                center_lat = (lat_top + lat_bottom) / 2.0
                center_lon = (lon_left + lon_right) / 2.0

                # Compute offsets around tile center (use local scale at center to stay consistent)
                m_per_deg_lat_c = meters_per_deg_lat(center_lat)
                m_per_deg_lon_c = meters_per_deg_lon(center_lat)

                north_m_tile = max(0.0, (lat_top - center_lat) * m_per_deg_lat_c)
                south_m_tile = max(0.0, (center_lat - lat_bottom) * m_per_deg_lat_c)
                west_m_tile = max(0.0, (center_lon - lon_left) * m_per_deg_lon_c)
                east_m_tile = max(0.0, (lon_right - center_lon) * m_per_deg_lon_c)

                # Numbered filename
                out_file = _numbered_filename(filename, piece_idx)

                # Recurse for this tile (turn off tiling to avoid re-splitting)
                tile_summary = generate_square_gpx_fast_vertical(
                    start_lat=center_lat,
                    start_lon=center_lon,
                    west_m=west_m_tile,
                    north_m=north_m_tile,
                    east_m=east_m_tile,
                    south_m=south_m_tile,
                    horizontal_step_m=horizontal_step_m,
                    secs_per_meter=secs_per_meter,
                    filename=out_file,
                    start_time_utc=start_time_utc,
                    max_columns=max_columns,
                    allow_exceed_east=allow_exceed_east,
                    max_tile_width_m=None,
                    max_tile_height_m=None,
                )
                # Ensure we collect the inner piece summary (not a tiled container)
                pieces.append(tile_summary if isinstance(tile_summary, dict) else {"file": out_file})
                piece_idx += 1

        # Combined summary
        combined = {
            "tiled": True,
            "original_file_base": filename,
            "pieces": pieces,
            "ul_corner": (lat_ul, lon_ul),
            "lr_corner": (lat_lr, lon_lr),
            "approx_total_width_m": width_m_est,
            "approx_total_height_m": height_m_est,
            "tile_count": len(pieces),
            "tile_dims_m": (tile_w, tile_h),
        }
        print(
            f"Tiled into {len(pieces)} piece(s). "
            f"UL=({lat_ul:.6f},{lon_ul:.6f}) LR=({lat_lr:.6f},{lon_lr:.6f}) "
            f"Total≈ {width_m_est:.1f}×{height_m_est:.1f} m"
        )
        return combined

    # --- Normal (non-tiling) single-file path below ---

    # Columns estimate and safety
    est_cols = max(1, int(math.ceil(width_m_est / max(1e-9, horizontal_step_m))))
    if est_cols > max_columns:
        raise RuntimeError(
            f"Estimated columns ~{est_cols:,} (> max_columns={max_columns:,}). "
            f"Increase horizontal_step_m or raise max_columns."
        )

    # Prepare GPX
    gpx = ET.Element(
        "gpx",
        version="1.1",
        creator="SquareGPXGeneratorFastVertical",
        xmlns="http://www.topografix.com/GPX/1/1"
    )
    trk = ET.SubElement(gpx, "trk")
    ET.SubElement(trk, "name").text = "Square coverage pattern (giant vertical steps)"
    trkseg = ET.SubElement(trk, "trkseg")

    def add_trkpt(lat: float, lon: float, t: datetime):
        trkpt = ET.SubElement(trkseg, "trkpt", lat=f"{lat:.7f}", lon=f"{lon:.7f}")
        ET.SubElement(trkpt, "ele").text = "0"
        ET.SubElement(trkpt, "time").text = t.strftime("%Y-%m-%dT%H:%M:%SZ")

    current_time = start_time_utc
    cur_lat, cur_lon = lat_ul, lon_ul
    add_trkpt(cur_lat, cur_lon, current_time)

    total_m = 0.0
    going_south = True

    def move_vertical(to_lat: float):
        """Single giant vertical segment to 'to_lat' at current longitude."""
        nonlocal cur_lat, cur_lon, current_time, total_m
        if cur_lat == to_lat:
            return
        avg_lat = (cur_lat + to_lat) / 2.0
        meters = abs(to_lat - cur_lat) * meters_per_deg_lat(avg_lat)
        total_m += meters
        current_time += timedelta(seconds=meters * secs_per_meter)
        cur_lat = to_lat
        add_trkpt(cur_lat, cur_lon, current_time)

    def step_east(allow_exceed: bool):
        """Step east by ~horizontal_step_m (single point)."""
        nonlocal cur_lat, cur_lon, current_time, total_m
        if horizontal_step_m == 0:
            return
        m_per_deg_lon_here = meters_per_deg_lon(cur_lat)
        desired_step_deg = horizontal_step_m / m_per_deg_lon_here
        next_lon = cur_lon + desired_step_deg
        if not allow_exceed and next_lon > lon_lr:
            next_lon = lon_lr
        meters = abs(next_lon - cur_lon) * meters_per_deg_lon(cur_lat)
        total_m += meters
        current_time += timedelta(seconds=meters * secs_per_meter)
        cur_lon = next_lon
        add_trkpt(cur_lat, cur_lon, current_time)

    # Build the pattern
    while True:
        # 1) Giant vertical move to the relevant boundary
        target_lat = lat_lr if going_south else lat_ul
        move_vertical(target_lat)

        # 2) If already at or beyond east boundary, stop
        if cur_lon >= lon_lr:
            break

        # 3) East step (possibly the last one)
        m_per_deg_lon_here = meters_per_deg_lon(cur_lat)
        would_next_lon = cur_lon + (horizontal_step_m / m_per_deg_lon_here)
        if would_next_lon > lon_lr:
            step_east(allow_exceed=allow_exceed_east)
            break
        else:
            step_east(allow_exceed=True)
            # 4) Toggle vertical direction for next column
            going_south = not going_south

    # Save GPX
    ET.ElementTree(gpx).write(filename, encoding="utf-8", xml_declaration=True)
    summary = {
        "file": filename,
        "ul_corner": (lat_ul, lon_ul),
        "lr_corner": (lat_lr, lon_lr),
        "approx_width_m": width_m_est,
        "approx_height_m": height_m_est,
        "columns_est": est_cols,
        "track_points": len(list(trkseg)),
        "distance_m": total_m
    }
    print(
        f"Saved GPX to {filename}\n"
        f" UL: ({lat_ul:.6f}, {lon_ul:.6f})\n"
        f" LR: ({lat_lr:.6f}, {lon_lr:.6f})\n"
        f" Width ≈ {width_m_est:.2f} m, Height ≈ {height_m_est:.2f} m\n"
        f" Columns ≈ {est_cols:,}\n"
        f" Track points: {summary['track_points']:,}\n"
        f" Total path ≈ {total_m:.1f} m"
    )
    return summary

In [12]:
#MAC
#start_lat=22.198745,   # Macau latitude
#start_lon=113.543873,  # Macau longitude
#west_m=2500, north_m=3000, # MAC
#east_m=5400, south_m=10500, # MAC
#horizontal_step_m=3.0,
#filename="MAC_square_solid.gpx"

#VAT 41°54′10″N，12°27′11″E）
#start_lat=41.902778,   # VAT latitude
#start_lon=12.453056,  # VAT longitude
#west_m=2000, north_m=2000, # VAT
#east_m=2000, south_m=2000, # VAT
#horizontal_step_m=3.0,
#filename="VAT_square_solid.gpx"

#MCO
#start_lat=43.7383,   
#start_lon=7.4244,  
#west_m=5000, north_m=5000,
#east_m=5000, south_m=5000,
#horizontal_step_m=3.0,
#filename="MCO_square_solid.gpx"
'''
# HKG example: very large rectangle — will be split into 10km x 10km tiles
generate_square_gpx_fast_vertical(
    start_lat=22.5058,
    start_lon=113.9448,
    west_m=25000, north_m=10000,
    east_m=52000, south_m=47000,
    horizontal_step_m=3.0,
    filename="HKG_square_solid.gpx",
    max_tile_width_m=10_000,   # 10 km wide tiles
    max_tile_height_m=10_000   # 10 km tall tiles
)
'''
#GIB
generate_square_gpx_fast_vertical(
    start_lat=36.15111, 
    start_lon=-5.34972,
    west_m=2600, north_m=7000,
    east_m=1700, south_m=5600,
    horizontal_step_m=3.0,
    filename="GIB_solid.gpx",
    max_tile_width_m=10000,   # 10 km wide tiles
    max_tile_height_m=15000   # 10 km tall tiles
)


Saved GPX to GIB_solid.gpx
 UL: (36.214195, -5.378612)
 LR: (36.100642, -5.330829)
 Width ≈ 4299.66 m, Height ≈ 12600.01 m
 Columns ≈ 1,434
 Track points: 2,869
 Total path ≈ 18072721.1 m


{'file': 'GIB_solid.gpx',
 'ul_corner': (36.214194784184464, -5.378611671270388),
 'lr_corner': (36.10064217265243, -5.330829291861669),
 'approx_width_m': 4299.655596963027,
 'approx_height_m': 12600.01330221364,
 'columns_est': 1434,
 'track_points': 2869,
 'distance_m': 18072721.07537492}

In [6]:
# HKG example: very large rectangle — will be split into 10km x 10km tiles
generate_square_gpx_fast_vertical(
    start_lat=22.5058,
    start_lon=113.9448,
    west_m=25000, north_m=10000,
    east_m=52000, south_m=47000,
    horizontal_step_m=3.0,
    filename="HKG_square_solid.gpx",
    max_tile_width_m=10_000,   # 10 km wide tiles
    max_tile_height_m=10_000   # 10 km tall tiles
)
# Outputs:
#   HKG_square_solid.1.gpx
#   HKG_square_solid.2.gpx
#   ...

Saved GPX to HKG_square_solid.1.gpx
 UL: (22.596104, 113.701827)
 LR: (22.505800, 113.799016)
 Width ≈ 9996.75 m, Height ≈ 10000.06 m
 Columns ≈ 3,333
 Track points: 6,667
 Total path ≈ 33340185.9 m
Saved GPX to HKG_square_solid.2.gpx
 UL: (22.596104, 113.799016)
 LR: (22.505800, 113.896205)
 Width ≈ 9996.75 m, Height ≈ 10000.06 m
 Columns ≈ 3,333
 Track points: 6,667
 Total path ≈ 33340185.9 m
Saved GPX to HKG_square_solid.3.gpx
 UL: (22.596104, 113.896205)
 LR: (22.505800, 113.993395)
 Width ≈ 9996.75 m, Height ≈ 10000.06 m
 Columns ≈ 3,333
 Track points: 6,667
 Total path ≈ 33340185.9 m
Saved GPX to HKG_square_solid.4.gpx
 UL: (22.596104, 113.993395)
 LR: (22.505800, 114.090584)
 Width ≈ 9996.75 m, Height ≈ 10000.06 m
 Columns ≈ 3,333
 Track points: 6,667
 Total path ≈ 33340185.9 m
Saved GPX to HKG_square_solid.5.gpx
 UL: (22.596104, 114.090584)
 LR: (22.505800, 114.187773)
 Width ≈ 9996.75 m, Height ≈ 10000.06 m
 Columns ≈ 3,333
 Track points: 6,667
 Total path ≈ 33340185.9 m
Saved

{'tiled': True,
 'original_file_base': 'HKG_square_solid.gpx',
 'pieces': [{'file': 'HKG_square_solid.1.gpx',
   'ul_corner': (22.596103932417606, 113.70182661095454),
   'lr_corner': (22.5058, 113.79901596657272),
   'approx_width_m': 9996.750430163318,
   'approx_height_m': 10000.056062936786,
   'columns_est': 3333,
   'track_points': 6667,
   'distance_m': 33340185.857768103},
  {'file': 'HKG_square_solid.2.gpx',
   'ul_corner': (22.596103932417606, 113.79901596657272),
   'lr_corner': (22.5058, 113.89620532219091),
   'approx_width_m': 9996.750430163318,
   'approx_height_m': 10000.056062936786,
   'columns_est': 3333,
   'track_points': 6667,
   'distance_m': 33340185.857768103},
  {'file': 'HKG_square_solid.3.gpx',
   'ul_corner': (22.596103932417606, 113.89620532219091),
   'lr_corner': (22.5058, 113.99339467780909),
   'approx_width_m': 9996.750430163318,
   'approx_height_m': 10000.056062936786,
   'columns_est': 3333,
   'track_points': 6667,
   'distance_m': 33340185.857768